In [ ]:
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import f
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import svm
from sklearn.svm import SVC
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
#from sklearn.metrics import plot_confusion_matrix


In [ ]:
data = pd.read_excel('../data/pd_data.xlsx')

In [ ]:
data.shape

In [ ]:
data.isna().sum().sort_values(ascending=False)

In [ ]:
data.dtypes

In [ ]:
data.columns

In [ ]:
num_cols = ['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)','MDVP:Jitter(%)', 
            'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer', 
            'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 
            'NHR', 'HNR', 'RPDE', 'DFA', 'spread1', 'spread2', 'D2', 'PPE', 'status']

In [ ]:
num_plots = len(num_cols)
num_rows = 6
no_cols = int(num_plots / num_rows) + (num_plots % num_rows > 0)
fig, axs = plt.subplots(nrows=num_rows, ncols=no_cols, figsize=(20, 15))

# iterate over the columns and plot a histogram for each numeric column
row = 0
col = 0
for column in num_cols:
    axs[row, col].hist(data[column], alpha=0.5, bins=20, color='orange')
    axs[row, col].set_xlabel(column)
    axs[row, col].set_ylabel('Frequency')
    if col == (no_cols - 1):
        row += 1
        col = 0
    else:
        col += 1

# show the plot
plt.show()

In [ ]:
num_plots = len(num_cols)
num_rows = 6
n_cols = 4
fig, axs = plt.subplots(nrows=num_rows, ncols=n_cols, figsize=(20, 22))
# iterate over the columns and plot a box plot for each numeric column
row = 0
col = 0
for column in num_cols:
    sns.boxplot(data=data[column], ax=axs[row, col], palette="pastel")
    axs[row, col].set_xlabel(column)
    axs[row, col].set_ylabel('Value')
    if col == (n_cols - 1):
        row += 1
        col = 0
    else:
        col += 1

# show the plot
plt.show()

In [ ]:
def f_test(df, independent_var, target_var):
    """
    Perform an F-test for equality of variances between two groups based on a binary target variable.

    Args:
        df (pandas.DataFrame): The input DataFrame.
        independent_var (str): The name of the column containing the independent variable.
        target_var (str): The name of the column containing the binary target variable.

    Returns:
        A tuple containing the F-statistic and the p-value for the F-test.
    """
    # Split the data into two groups based on the binary target variable
    group1 = df.loc[df[target_var] == 0, independent_var]
    group2 = df.loc[df[target_var] == 1, independent_var]

    # Calculate the variances of the independent variable for each group
    var1 = group1.var(ddof=1)
    var2 = group2.var(ddof=1)

    # Calculate the F-statistic
    f_stat = var1 / var2

    # Calculate the degrees of freedom for the F-distribution
    df1 = len(group1) - 1
    df2 = len(group2) - 1

    # Calculate the p-value for the F-test
    p_value = f.sf(f_stat, df1, df2)

    # Return the F-statistic and the p-value
    return f_stat, p_value


ftest_results = pd.DataFrame(columns=['Independent Variable', 'F-Statistic', 'p-value'])

# Iterate over the columns in the DataFrame and apply the f_test function to each one
for col in num_cols:
    f_stat, p_value = f_test(data, col, 'status')
    ftest_results = pd.concat([ftest_results, pd.DataFrame([{'Independent Variable': col, 'F-Statistic': f_stat, 'p-value': p_value}])], ignore_index=True)

# Print the results table
print(ftest_results)


In [ ]:
corr_matrix = data[num_cols].corr()

# compute the p-values for the correlation coefficients
p_values = data[num_cols].corr(method=lambda x, y: stats.pearsonr(x, y)[1].round(3)) - np.eye(len(num_cols))

In [ ]:
plt.figure(figsize=(20,20))
plot = sns.heatmap(p_values, annot = corr_matrix, center = 0.05)

In [ ]:
plt.figure(figsize=(30,20))
plot = sns.heatmap(corr_matrix, annot = p_values)

In [ ]:
variation_freq = ['MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP']
variation_amp = ['MDVP:Shimmer', 'MDVP:Shimmer(dB)','Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA']
fund_freq_vars = ['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'status']

In [ ]:
frequency_corr_matrix = data[variation_freq].corr()

# compute the p-values for the correlation coefficients
frequency_p_values = data[variation_freq].corr(method=lambda x, y: stats.pearsonr(x, y)[1].round(3)) - np.eye(len(variation_freq))

plt.figure(figsize=(10,5))
plot = sns.heatmap(frequency_p_values, annot = frequency_corr_matrix, center = 0.05)

In [ ]:
amplitude_corr_matrix = data[variation_amp].corr()

# compute the p-values for the correlation coefficients
amplitude_p_values = data[variation_amp].corr(method=lambda x, y: stats.pearsonr(x, y)[1].round(3)) - np.eye(len(variation_amp))

plt.figure(figsize=(10,5))
plot = sns.heatmap(amplitude_p_values, annot = amplitude_corr_matrix, center = 0.05)

In [ ]:
print(amplitude_p_values)
print(frequency_p_values)

In [ ]:
fundfreq_corr_matrix = data[fund_freq_vars].corr()

# compute the p-values for the correlation coefficients
fundfreq_p_values = data[fund_freq_vars].corr(method=lambda x, y: stats.pearsonr(x, y)[1].round(3)) - np.eye(len(fund_freq_vars))

plt.figure(figsize=(5,5))
plot = sns.heatmap(fundfreq_p_values, annot = fundfreq_corr_matrix, center = 0.05)

In [ ]:
num_plots = len(num_cols)
num_rows = 5
n_cols = 5
fig, axs = plt.subplots(nrows=num_rows, ncols=n_cols, figsize=(35, 35))
# iterate over the columns and plot a box plot for each numeric column
row = 0
col = 0
for column in num_cols:
    sns.boxplot(x='status',y=column, data=data, ax=axs[row, col], palette="pastel")
    axs[row, col].set_xlabel(column)
    axs[row, col].set_ylabel('Value')
    if col == (n_cols - 1):
        row += 1
        col = 0
    else:
        col += 1

# show the plot
plt.show()

In [ ]:
def count_outliers_iqr(column):
    q1 = np.percentile(column, 25)
    q3 = np.percentile(column, 75)
    iqr = q3 - q1
    upper_threshold = q3 + 1.5 * iqr
    lower_threshold = q1 - 1.5 * iqr
    num_outliers = sum((column > upper_threshold) | (column < lower_threshold))
    return num_outliers

for col in num_cols:
    num_outliers = count_outliers_iqr(data[col])
    print(f"Number of outliers in {col}: {num_outliers}")

As the Parkinson's disease causes a change in variance in fundamental frequency, we tried to calculate a proxy for spread in the Fundamental frequency.

In [ ]:
data['MDVP:F_Spd'] = data['MDVP:Fhi(Hz)'] - data['MDVP:Flo(Hz)']

In [ ]:
df = data
df.columns

In [ ]:
df = df.drop(['name','status'], axis = 1)

In [ ]:
df.head(10)

In [ ]:
df.columns

Scaling/Normalising the data using the standard scaler.

In [ ]:
# Standardize the data in the new dataframe
scaler = StandardScaler()
df_scale = scaler.fit_transform(df)

df_scaled = pd.DataFrame(df_scale, columns=['MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 
                                            'MDVP:Jitter(%)','MDVP:Jitter(Abs)', 'MDVP:RAP', 
                                            'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer', 
                                            'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5',
                                            'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE', 'DFA', 
                                            'spread1', 'spread2', 'D2', 'PPE', 'MDVP:F_Spd'])

As there are multiple variables which capture a variation of the Jitter and Shimmer measures, we will drop them to reduce collinearity.

In [ ]:
df_scaled = df_scaled.drop(['MDVP:Jitter(%)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 
                          'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 
                          'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)'], axis = 1)

In [ ]:
df_scaled.columns

In [ ]:
df_scaled['status'] = data['status'].copy()

In [ ]:
corr_matrix = df_scaled.corr()

# compute the p-values for the correlation coefficients
p_values = df_scaled.corr(method=lambda x, y: stats.pearsonr(x, y)[1].round(3)) - np.eye(len(df_scaled.columns))

In [ ]:
plt.figure(figsize=(20,20))
plot = sns.heatmap(p_values, annot = corr_matrix, center = 0.05)

In [ ]:
ftest_scaled_results = pd.DataFrame(columns=['Independent Variable', 'F-Statistic', 'p-value'])
scale_cols = ['MDVP:Fo(Hz)', 'MDVP:Jitter(Abs)', 'MDVP:Shimmer', 'NHR', 'HNR', 'RPDE','DFA', 
              'spread1', 'spread2', 'D2', 'PPE', 'MDVP:F_Spd']

# Iterate over the columns in the DataFrame and apply the f_test function to each one
for col in scale_cols:
    f_stat, p_value = f_test(df_scaled, col, 'status')
    ftest_scaled_results = pd.concat([ftest_scaled_results, pd.DataFrame([{'Independent Variable': col, 'F-Statistic': f_stat, 'p-value': p_value}])], ignore_index=True)

# Print the results table
print(ftest_scaled_results)


In [ ]:
# Create a grid of 3x4 subplots for the Q-Q plots
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(12, 12))

# Flatten the array of subplots for easier iteration
axes = axes.flatten()

# Iterate over each variable in the DataFrame and plot its Q-Q plot on a subplot
for i, col in enumerate(df_scaled.columns):
    sm.graphics.qqplot(df_scaled[col], line='s', ax=axes[i])
    axes[i].set_title(col)

# Remove any extra subplots
for i in range(len(df_scaled.columns), len(axes)):
    fig.delaxes(axes[i])

# Adjust the spacing between subplots and display the plot
fig.tight_layout()
plt.show()

Implementing Ridge regression

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Define the independent and dependent variables
X = df_scaled.drop(['status'], axis=1)
y = df_scaled['status']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=890)

# Instantiate the Ridge regression model
ridge_model = Ridge(alpha=1.0)

# Fit the model on the training data
ridge_model.fit(X_train, y_train)

# Predict the target variable on the test data
y_pred = ridge_model.predict(X_test)

# Evaluate the model performance
mse = mean_squared_error(y_test, y_pred)
print(f"Mean squared error: {mse}")

In [ ]:
from sklearn.linear_model import RidgeCV

# Create a RidgeCV object
ridgecv = RidgeCV(alphas=[0.01, 0.1, 1.0, 5, 10.0, 15, 25, 50, 100])

# Fit the RidgeCV object to the data
ridgecv.fit(X_train, y_train)

# Print the optimal alpha value
print(f"Optimal alpha value: {ridgecv.alpha_}")


In [ ]:
# Instantiate the Ridge regression model
ridge25_model = Ridge(alpha=25.0)

# Fit the model on the training data
ridge25_model.fit(X_train, y_train)

# Predict the target variable on the test data
y_pred_25 = ridge25_model.predict(X_test)

# Evaluate the model performance
mse_25 = mean_squared_error(y_test, y_pred_25)
print(f"Mean squared error: {mse_25}")

In [ ]:
from sklearn.metrics import r2_score

r2_25 = r2_score(y_test, y_pred_25)

# calculate the adjusted R-squared value
n = len(X_test)
k = X_test.shape[1]
adj_r2_25 = 1 - ((1 - r2_25) * (n - 1) / (n - k - 1))

print(f"R-squared for a = 25: {r2_25:.4f}")
print(f"Adjusted R-squared for a = 25: {adj_r2_25:.4f}")

In [ ]:
r2 = r2_score(y_test, y_pred)

# calculate the adjusted R-squared value
n = len(X_test)
k = X_test.shape[1]
adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - k - 1))

print(f"R-squared: {r2:.4f}")
print(f"Adjusted R-squared: {adj_r2:.4f}")

In [ ]:
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
predDT = clf.predict(X_test)

print(classification_report(y_test, predDT))

In [ ]:
param_grid = { 
    'max_depth' :range(1,10),
    'random_state':range(30,210,30),
    'criterion' :['gini', 'entropy']
}
CV_dt = GridSearchCV(estimator=clf, param_grid=param_grid, cv= 5)
CV_dt.fit(X_train, y_train)

In [ ]:
CV_dt.best_params_

In [ ]:
dt1=DecisionTreeClassifier(random_state=30, max_depth=1, criterion='gini')
dt1.fit(X_train, y_train)
predDT = dt1.predict(X_test) 
print(classification_report(y_test, predDT))

In [ ]:
y_pred_proba = dt1.predict_proba(X_test)[::,1]
fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred_proba)
auc = metrics.roc_auc_score(y_test, y_pred_proba)
plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.legend(loc=4)
plt.show()

### Random forest

In [ ]:
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)
predRF = rfc.predict(X_test)

print(classification_report(y_test, predRF))

In [ ]:
param_grid = { 
    'n_estimators': range(100,300,25),
    'max_depth' :range(1,10),
    'random_state':range(100,250,50),
    'criterion' :['gini', 'entropy']
}
CV_rfc = GridSearchCV(estimator=rfc, param_grid=param_grid, cv= 5)
CV_rfc.fit(X_train, y_train)

In [ ]:
CV_rfc.best_params_

In [ ]:
rfc1=RandomForestClassifier(random_state=100, n_estimators= 250, max_depth=9, criterion='entropy')
rfc1.fit(X_train, y_train)
predRFC = rfc1.predict(X_test)
print(classification_report(y_test, predRFC))

In [ ]:
y_pred_proba = rfc1.predict_proba(X_test)[::,1]
fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred_proba)
auc = metrics.roc_auc_score(y_test, y_pred_proba)
plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.legend(loc=4)
plt.show()

In [ ]:
# Get feature importances
importances = rfc1.feature_importances_

# Print feature importances
for feature, importance in zip(X.columns, importances):
    print(f"{feature}: {importance}")

### SVM

In [ ]:
grid_SVC = svm.SVC(C = 100, gamma = 1, kernel = 'linear')
grid_SVC.fit(X_train, y_train)



predSVC = grid_SVC.predict(X_test) 
  
# print classification report 
print("\n", classification_report(y_test, predSVC))

In [ ]:
param_grid = {'kernel':['linear','rbf','poly'],'C': [0.5, 1, 10, 100],  
              'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}

grid_SVC = GridSearchCV(svm.SVC(), param_grid, scoring='f1', verbose = 3)
grid_SVC.fit(X_train, y_train)

# print best parameter after tuning 
print("\nBest Parameters: ", grid_SVC.best_params_)

In [ ]:
# print how our model looks after hyper-parameter tuning
print("\n", grid_SVC.best_estimator_)

predSVC = grid_SVC.predict(X_test) 
  
# print classification report 
print("\n", classification_report(y_test, predSVC))

In [ ]:
fpr, tpr, _ = metrics.roc_curve(y_test,  predSVC)
auc = metrics.roc_auc_score(y_test, predSVC)
plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.legend(loc=4)
plt.show()

### KNN

In [ ]:
Ks = 10
mean_acc = []
ConfustionMx = [];
for n in range(2,Ks):
    
    #Train Model and Predict  
    neigh = KNeighborsClassifier(n_neighbors = n).fit(X_train,y_train)
    yhat=neigh.predict(X_test)
    mean_acc.append(metrics.accuracy_score(y_test, yhat))  
print('Neighbor Accuracy List')
print(mean_acc)

In [ ]:
plt.plot(range(2,Ks),mean_acc,'g')
plt.ylabel('Accuracy ')
plt.xlabel('Number of Neighbours (K)')
plt.tight_layout()
plt.show()

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
predKNN = knn.predict(X_test)
print(classification_report(y_test, yhat))

In [ ]:
y_pred_proba = knn.predict_proba(X_test)[::,1]
fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred_proba)
auc = metrics.roc_auc_score(y_test, y_pred_proba)
plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.legend(loc=4)
plt.show()

In [ ]:
from sklearn.metrics import precision_score,recall_score ,accuracy_score, f1_score, r2_score, log_loss

chart = {
        'Metric':["Accuracy", "F1-Score", "Recall", "Precision", "R2-Score"],
        'DT':[accuracy_score(y_test, predDT), f1_score(y_test, predDT), recall_score(y_test, predDT), precision_score(y_test, predDT), r2_score(y_test, predDT)],
        'RF':[accuracy_score(y_test, predRF), f1_score(y_test, predRFC), recall_score(y_test, predRFC), precision_score(y_test, predRFC), r2_score(y_test, predRFC)],
        'SVM':[accuracy_score(y_test, predSVC), f1_score(y_test, predSVC), recall_score(y_test, predSVC), precision_score(y_test, predSVC), r2_score(y_test, predSVC)],
        'KNN':[accuracy_score(y_test, predKNN), f1_score(y_test, predKNN), recall_score(y_test, predKNN), precision_score(y_test, predKNN), r2_score(y_test, predKNN)],
}
chart = pd.DataFrame(chart)
display(chart)

## Repeating Analysis on Principal component data

In [ ]:
df_pc = data

In [ ]:
df_pc.columns

In [ ]:
df_pc = df_pc.drop(['name'], axis = 1)

In [ ]:
df_pc1 = df_pc[variation_freq]

In [ ]:
df_pc1 = scaler.fit_transform(df_pc1)

Performing principal component analysis on the Jitter variables

In [ ]:
pca = PCA()
pca.fit(df_pc1)
explained_variance_ratio = pca.explained_variance_ratio_
print(explained_variance_ratio)

In [ ]:
cumulative_explained_variance_ratio = np.cumsum(explained_variance_ratio)
num_components = np.argmax(cumulative_explained_variance_ratio >= 0.95) + 1
print(num_components)

In [ ]:
eigenvalues = pca.explained_variance_
print(eigenvalues)

In [ ]:
plt.plot(np.arange(1, len(eigenvalues)+1), eigenvalues, 'o-')
plt.xlabel('Principal Component')
plt.ylabel('Eigenvalue')
plt.title('Scree Plot')
plt.show()

In [ ]:
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
plt.plot(np.arange(1, len(cumulative_variance_ratio)+1), cumulative_variance_ratio, 'o-')
plt.xlabel('Principal Component')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.title('Cumulative Explained Variance Ratio Plot')
plt.show()

num_components = np.argmax(cumulative_variance_ratio >= 0.95) + 1
print(num_components)


Only one principal component is necessary for explaining the 5 variables

In [ ]:
pca = PCA(n_components=num_components)
data_transformed = pca.fit_transform(df_pc1)

In [ ]:
data_transformed = pd.DataFrame(data_transformed, columns = ['Jitter_pc'])
df_pc1 = pd.DataFrame(df_pc1, columns = ['MDVP:Jitter(%)','MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP'])

In [ ]:
data_transformed

In [ ]:
df_pc['Jitter_pc'] = data_transformed['Jitter_pc']

In [ ]:
df_pc.columns

Performing PCA on Shimmer variables

In [ ]:
pca2 = PCA()
df_pc2 = df_pc[variation_amp]
df_pc2 = scaler.fit_transform(df_pc2)

pca2.fit(df_pc2)
explained_variance_ratio = pca2.explained_variance_ratio_
print(explained_variance_ratio)

In [ ]:
cumulative_explained_variance_ratio = np.cumsum(explained_variance_ratio)
num_components = np.argmax(cumulative_explained_variance_ratio >= 0.95) + 1
print(num_components)

In [ ]:
eigenvalues = pca2.explained_variance_
print(eigenvalues)

In [ ]:
plt.plot(np.arange(1, len(eigenvalues)+1), eigenvalues, 'o-')
plt.xlabel('Principal Component')
plt.ylabel('Eigenvalue')
plt.title('Scree Plot')
plt.show()

In [ ]:
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
plt.plot(np.arange(1, len(cumulative_variance_ratio)+1), cumulative_variance_ratio, 'o-')
plt.xlabel('Principal Component')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.title('Cumulative Explained Variance Ratio Plot')
plt.show()

num_components = np.argmax(cumulative_variance_ratio >= 0.95) + 1
print(num_components)

Similarly, only one principal component required for Simmer variables as well

In [ ]:
pca2 = PCA(n_components=num_components)
data_transformed2 = pca2.fit_transform(df_pc2)

In [ ]:
data_transformed2 = pd.DataFrame(data_transformed2, columns = ['Shimmer_pc'])
df_pc2 = pd.DataFrame(df_pc2, columns = ['MDVP:Shimmer', 'MDVP:Shimmer(dB)','Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA'])

In [ ]:
df_pc['Shimmer_pc'] = data_transformed2['Shimmer_pc']

In [ ]:
df_pc = df_pc.drop(['MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)',
                    'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP',
                    'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5',
                    'MDVP:APQ', 'Shimmer:DDA'], axis = 1)

In [ ]:
df_pc.head(7)

Splitting into test and train

In [ ]:
X = df_pc.drop(['status'], axis=1)
y = df_pc['status']

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X, y , test_size=0.2, random_state=20)

### Decision Tree

In [ ]:
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
predDT = clf.predict(X_test)

print(classification_report(y_test, predDT))

In [ ]:
param_grid = { 
    'max_depth' :range(1,10),
    'random_state':range(30,210,30),
    'criterion' :['gini', 'entropy']
}
CV_dt = GridSearchCV(estimator=clf, param_grid=param_grid, cv= 5)
CV_dt.fit(X_train, y_train)

In [ ]:
CV_dt.best_params_

In [ ]:
dt1=DecisionTreeClassifier(random_state=30, max_depth=1, criterion='gini')
dt1.fit(X_train, y_train)
predDT = dt1.predict(X_test) 
print(classification_report(y_test, predDT))

In [ ]:
y_pred_proba = dt1.predict_proba(X_test)[::,1]
fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred_proba)
auc = metrics.roc_auc_score(y_test, y_pred_proba)
plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.legend(loc=4)
plt.show()

### Random forest

In [ ]:
rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)
predRF = rfc.predict(X_test)

print(classification_report(y_test, predRF))

In [ ]:
param_grid = { 
    'n_estimators': range(100,300,25),
    'max_depth' :range(1,10),
    'random_state':range(100,250,50),
    'criterion' :['gini', 'entropy']
}
CV_rfc = GridSearchCV(estimator=rfc, param_grid=param_grid, cv= 5)
CV_rfc.fit(X_train, y_train)

In [ ]:
CV_rfc.best_params_

In [ ]:
rfc1=RandomForestClassifier(random_state=150, n_estimators= 275, max_depth=6, criterion='gini')
rfc1.fit(X_train, y_train)
predRFC = rfc1.predict(X_test)
print(classification_report(y_test, predRFC))

In [ ]:
y_pred_proba = rfc1.predict_proba(X_test)[::,1]
fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred_proba)
auc = metrics.roc_auc_score(y_test, y_pred_proba)
plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.legend(loc=4)
plt.show()

In [ ]:
# Get feature importances
importances = rfc1.feature_importances_

# Print feature importances
for feature, importance in zip(X.columns, importances):
    print(f"{feature}: {importance}")

### SVM

In [ ]:
grid_SVC = svm.SVC(C = 100, gamma = 1, kernel = 'linear')
grid_SVC.fit(X_train, y_train)



predSVC = grid_SVC.predict(X_test) 
  
# print classification report 
print("\n", classification_report(y_test, predSVC))

In [ ]:
param_grid = {'kernel':['linear','rbf'],'C': [0.5, 1, 10, 100],  
              'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}

grid_SVC = GridSearchCV(svm.SVC(), param_grid, scoring='f1', verbose = 3)
grid_SVC.fit(X_train, y_train)

# print best parameter after tuning 
print("\nBest Parameters: ", grid_SVC.best_params_)

In [ ]:
# print how our model looks after hyper-parameter tuning
print("\n", grid_SVC.best_estimator_)

predSVC = grid_SVC.predict(X_test) 
  
# print classification report 
print("\n", classification_report(y_test, predSVC))

In [ ]:
fpr, tpr, _ = metrics.roc_curve(y_test,  predSVC)
auc = metrics.roc_auc_score(y_test, predSVC)
plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.legend(loc=4)
plt.show()

### KNN

In [ ]:
Ks = 10
mean_acc = []
ConfustionMx = [];
for n in range(2,Ks):
    
    #Train Model and Predict  
    neigh = KNeighborsClassifier(n_neighbors = n).fit(X_train,y_train)
    yhat=neigh.predict(X_test)
    mean_acc.append(metrics.accuracy_score(y_test, yhat))  
print('Neighbor Accuracy List')
print(mean_acc)

In [ ]:
plt.plot(range(2,Ks),mean_acc,'g')
plt.ylabel('Accuracy ')
plt.xlabel('Number of Neighbours (K)')
plt.tight_layout()
plt.show()

In [ ]:
knn = KNeighborsClassifier(n_neighbors=2)
knn.fit(X_train, y_train)
predKNN = knn.predict(X_test)
print(classification_report(y_test, yhat))

In [ ]:
y_pred_proba = knn.predict_proba(X_test)[::,1]
fpr, tpr, _ = metrics.roc_curve(y_test,  y_pred_proba)
auc = metrics.roc_auc_score(y_test, y_pred_proba)
plt.plot(fpr,tpr,label="data 1, auc="+str(auc))
plt.legend(loc=4)
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance
result = permutation_importance(knn, X_test, y_test, n_repeats=30, random_state=0)

# Print feature importances
for feature, importance in zip(X.columns, result.importances_mean):
    print(f"{feature}: {importance}")

In [ ]:
from sklearn.metrics import precision_score,recall_score ,accuracy_score, f1_score, r2_score, log_loss

chart = {
        'Metric':["Accuracy", "F1-Score", "Recall", "Precision", "R2-Score"],
        'DT':[accuracy_score(y_test, predDT), f1_score(y_test, predDT), recall_score(y_test, predDT), precision_score(y_test, predDT), r2_score(y_test, predDT)],
        'RF':[accuracy_score(y_test, predRF), f1_score(y_test, predRFC), recall_score(y_test, predRFC), precision_score(y_test, predRFC), r2_score(y_test, predRFC)],
        'SVM':[accuracy_score(y_test, predSVC), f1_score(y_test, predSVC), recall_score(y_test, predSVC), precision_score(y_test, predSVC), r2_score(y_test, predSVC)],
        'KNN':[accuracy_score(y_test, predKNN), f1_score(y_test, predKNN), recall_score(y_test, predKNN), precision_score(y_test, predKNN), r2_score(y_test, predKNN)],
}
chart = pd.DataFrame(chart)
display(chart)